# 📈 Notebook 04 — Model Evaluation & Scorecard
## ROC | KS | Gini | PSI | Score Bands | Calibration

Deep-dive evaluation of the trained credit scoring models,
with Basel II/III compliance metrics, score band analysis,
and adverse action reporting.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns, warnings, sys, os
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
print("✅ Evaluation notebook ready — run after Notebook 03 to load trained models.")
print("   For demo, generating synthetic predictions that match expected model performance.")

np.random.seed(42)
n_test = 10_000
y_test = np.random.binomial(1, 0.12, n_test)

# Simulate realistic model predictions
cw = 1 - (y_test * 0.6 + np.random.normal(0.3, 0.18, n_test))
nn_proba  = np.clip(cw + np.random.normal(0, 0.07, n_test), 0.01, 0.99)
xgb_proba = np.clip(cw + np.random.normal(0, 0.08, n_test), 0.01, 0.99)
ensemble  = 0.55 * nn_proba + 0.45 * xgb_proba

print(f"NN AUC:  {roc_auc_score(y_test, nn_proba):.4f}")
print(f"XGB AUC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"Ens AUC: {roc_auc_score(y_test, ensemble):.4f}")


## 📊 KS Statistic Deep Dive

In [ ]:
# KS Statistic
def ks_chart(y_true, y_score, ax, title, colour):
    sort_idx = np.argsort(y_score)[::-1]
    pct = np.linspace(0, 1, len(y_true))
    cum_bad  = np.cumsum(y_true[sort_idx]) / y_true.sum()
    cum_good = np.cumsum(1-y_true[sort_idx]) / (1-y_true).sum()
    ks = (cum_bad - cum_good).max()
    ks_idx = (cum_bad - cum_good).argmax()
    
    ax.fill_between(pct, cum_bad, cum_good, alpha=0.12, color=colour)
    ax.plot(pct, cum_bad,  color="#C0392B", lw=2.5, label="Bad CDF")
    ax.plot(pct, cum_good, color="#27AE60", lw=2.5, label="Good CDF")
    ax.annotate(f"KS = {ks:.4f}", xy=(pct[ks_idx], cum_bad[ks_idx]),
                xytext=(pct[ks_idx]+0.05, cum_bad[ks_idx]-0.1),
                fontsize=11, fontweight="bold", color=colour,
                arrowprops=dict(arrowstyle="->", color=colour))
    ax.axvline(pct[ks_idx], color=colour, ls="--", lw=1.5, alpha=0.7)
    ax.set(title=f"{title} | KS={ks:.4f}", xlabel="Population %", ylabel="Cumulative %")
    ax.legend(fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("KS Statistic — Model Separation Power", fontsize=13, fontweight="bold")
ks_chart(y_test, nn_proba, axes[0], "Neural Network", "#2E86C1")
ks_chart(y_test, ensemble, axes[1], "Ensemble (NN+XGB)", "#1B4F72")
plt.tight_layout()
plt.savefig("../reports/figures/10_ks_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 🎯 Score Band Calibration

In [ ]:
# Score calibration by band
scores_all = 300 + 550 * (1 - ensemble)

bands = [(300,499,"Very High Risk","#C0392B"),
         (500,599,"High Risk","#E67E22"),
         (600,699,"Medium Risk","#F39C12"),
         (700,774,"Low Risk","#27AE60"),
         (775,850,"Very Low Risk","#1ABC9C")]

cal_df = []
for lo, hi, name, col in bands:
    mask = (scores_all >= lo) & (scores_all <= hi)
    if mask.sum() > 0:
        cal_df.append({
            "Band": name, "Score Range": f"{lo}–{hi}",
            "Population %": f"{mask.mean()*100:.1f}%",
            "Default Rate": f"{y_test[mask].mean()*100:.1f}%",
            "Count": mask.sum(),
            "Decision": ["Decline","Manual Review","Conditional","Approve","Preferred"][len(cal_df)]
        })

cal_df = pd.DataFrame(cal_df)
print("Score Band Calibration Report")
print("=" * 70)
print(cal_df.to_string(index=False))


## 📋 Regulatory Compliance Metrics

In [ ]:
print("\nRegulatory Credit Model Validation Report")
print("=" * 60)
auc = roc_auc_score(y_test, ensemble)
gini = 2 * auc - 1
ks_stat, _ = ks_2samp(ensemble[y_test==0], ensemble[y_test==1])

metrics_table = {
    "AUC-ROC": (auc, 0.75, "Discriminatory Power"),
    "Gini Coefficient": (gini, 0.50, "Industry Standard Separator"),
    "KS Statistic": (ks_stat, 0.35, "Population Separation"),
}

for metric, (val, threshold, desc) in metrics_table.items():
    status = "✅ PASS" if val >= threshold else "❌ FAIL"
    print(f"  {metric:<22} {val:.4f}  (threshold: {threshold:.2f})  {status}  — {desc}")

print(f"\n  Model Status: {'✅ APPROVED FOR DEPLOYMENT' if auc >= 0.75 else '❌ REQUIRES IMPROVEMENT'}")
